In [9]:
using DataFrames
using RDatasets
using Statistics

In [5]:
iris = dataset("datasets", "iris")
first(iris, 3)

,SepalLength,SepalWidth,PetalLength,PetalWidth,Species
,Float64,Float64,Float64,Float64,Cat…
1,5.1,3.5,1.4,0.2,setosa
2,4.9,3.0,1.4,0.2,setosa
3,4.7,3.2,1.3,0.2,setosa


In [6]:
gdf = groupby(iris, :Species)

,SepalLength,SepalWidth,PetalLength,PetalWidth,Species
,Float64,Float64,Float64,Float64,Cat…
1,5.1,3.5,1.4,0.2,setosa
2,4.9,3.0,1.4,0.2,setosa
3,4.7,3.2,1.3,0.2,setosa
4,4.6,3.1,1.5,0.2,setosa
5,5.0,3.6,1.4,0.2,setosa
6,5.4,3.9,1.7,0.4,setosa
7,4.6,3.4,1.4,0.3,setosa
8,5.0,3.4,1.5,0.2,setosa
9,4.4,2.9,1.4,0.2,setosa


In [10]:
combine(gdf, :PetalLength => mean)

,Species,PetalLength_mean
,Cat…,Float64
1,setosa,1.462
2,versicolor,4.26
3,virginica,5.552


In [12]:
combine(gdf, nrow)

,Species,nrow
,Cat…,Int64
1,setosa,50
2,versicolor,50
3,virginica,50


In [13]:
combine(gdf, nrow, :PetalLength => mean)

,Species,nrow,PetalLength_mean
,Cat…,Int64,Float64
1,setosa,50,1.462
2,versicolor,50,4.26
3,virginica,50,5.552


In [14]:
combine(gdf, nrow, :PetalLength => mean => :mean)

,Species,nrow,mean
,Cat…,Int64,Float64
1,setosa,50,1.462
2,versicolor,50,4.26
3,virginica,50,5.552


In [15]:
combine(
    [:PetalLength, :SepalLength] => 
        (p, s) -> (a=mean(p)/mean(s), b=sum(p)),
    gdf)

,Species,a,b
,Cat…,Float64,Float64
1,setosa,0.29205,73.1
2,versicolor,0.717655,213.0
3,virginica,0.842744,277.6


In [19]:
combine(
    gdf, 
    AsTable([:PetalLength, :SepalLength]) => 
        x -> std(x.PetalLength) / std(x.SepalLength))

,Species,PetalLength_SepalLength_function
,Cat…,Float64
1,setosa,0.492678
2,versicolor,0.910378
3,virginica,0.867923


In [20]:
combine(x -> std(x.PetalLength) / std(x.SepalLength), gdf)

,Species,x1
,Cat…,Float64
1,setosa,0.492678
2,versicolor,0.910378
3,virginica,0.867923


In [21]:
combine(gdf, 1:2 => cor, nrow)

,Species,SepalLength_SepalWidth_cor,nrow
,Cat…,Float64,Int64
1,setosa,0.742547,50
2,versicolor,0.525911,50
3,virginica,0.457228,50


In [22]:
select(gdf, 1:2 => cor)

,Species,SepalLength_SepalWidth_cor
,Cat…,Float64
1,setosa,0.742547
2,setosa,0.742547
3,setosa,0.742547
4,setosa,0.742547
5,setosa,0.742547
6,setosa,0.742547
7,setosa,0.742547
8,setosa,0.742547
9,setosa,0.742547


In [25]:
transform(gdf, :Species => x -> chop.(x, head=5, tail=0))

,Species,SepalLength,SepalWidth,PetalLength,PetalWidth,Species_function
,Cat…,Float64,Float64,Float64,Float64,SubStri…
1,setosa,5.1,3.5,1.4,0.2,a
2,setosa,4.9,3.0,1.4,0.2,a
3,setosa,4.7,3.2,1.3,0.2,a
4,setosa,4.6,3.1,1.5,0.2,a
5,setosa,5.0,3.6,1.4,0.2,a
6,setosa,5.4,3.9,1.7,0.4,a
7,setosa,4.6,3.4,1.4,0.3,a
8,setosa,5.0,3.4,1.5,0.2,a
9,setosa,4.4,2.9,1.4,0.2,a


In [26]:
combine(gdf) do df
    (m=mean(df.PetalLength), s_sq=var(df.PetalLength))
end

,Species,m,s_sq
,Cat…,Float64,Float64
1,setosa,1.462,0.0301592
2,versicolor,4.26,0.220816
3,virginica,5.552,0.304588


In [27]:
for subdf in groupby(iris, :Species)
    println(size(subdf, 1))
end

50
50
50


In [28]:
for (key, subdf) in pairs(groupby(iris, :Species))
    println("N rows for $(key.Species): $(nrow(subdf))")
end

N rows for setosa: 50
N rows for versicolor: 50
N rows for virginica: 50


In [29]:
df = DataFrame(g=repeat(1:1000, inner=5), x=1:5000)
gdf = groupby(df, :g)

,g,x
,Int64,Int64
1,1,1
2,1,2
3,1,3
4,1,4
5,1,5
,g,x
,Int64,Int64
1,1000,4996
2,1000,4997


In [30]:
gd = groupby(iris, :Species)
combine(gd, valuecols(gd) .=> mean)

,Species,SepalLength_mean,SepalWidth_mean,PetalLength_mean,PetalWidth_mean
,Cat…,Float64,Float64,Float64,Float64
1,setosa,5.006,3.428,1.462,0.246
2,versicolor,5.936,2.77,4.26,1.326
3,virginica,6.588,2.974,5.552,2.026


In [32]:
combine(
    gd, 
    valuecols(gd) .=> (x -> (x .- mean(x)) / std(x)) .=> valuecols(gd))

,Species,SepalLength,SepalWidth,PetalLength,PetalWidth
,Cat…,Float64,Float64,Float64,Float64
1,setosa,0.266674,0.189941,-0.357011,-0.436492
2,setosa,-0.300718,-1.1291,-0.357011,-0.436492
3,setosa,-0.868111,-0.601481,-0.932836,-0.436492
4,setosa,-1.15181,-0.865288,0.218813,-0.436492
5,setosa,-0.0170218,0.453749,-0.357011,-0.436492
6,setosa,1.11776,1.24517,1.37046,1.4613
7,setosa,-1.15181,-0.0738661,-0.357011,0.512404
8,setosa,-0.0170218,-0.0738661,0.218813,-0.436492
9,setosa,-1.7192,-1.3929,-0.357011,-0.436492
